# Embeddings Playground — coding workshop (beginner)

**Expected time:** ~15-20 minutes, individual, no GPU needed (both parts run fine on
CPU; the downloads take a minute or two).

In the lesson, the toy vectors aunt = (4, 2) and uncle = (3, 3) had a cosine similarity of
about 0.95. Here you compute cosine similarities with *real*, pretrained embeddings, then
ask a sharper question: in a real multilingual model, do
Galician words actually look distinct, or do they blur into Spanish and Portuguese?

## How this workshop works

This is the coding version of the notebook, **beginner level**. Some code cells have gaps
marked `___`. An exercise above each such cell says what to write. This version takes
longer than the expected time given above.

1. Read the exercise. Open the hints if you need them.
2. Replace every `___` in the cell with your code, then run the cell.
3. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
4. If you are stuck, the solution is below the check cell. Open it after you have tried.

A cell that still contains `___` stops with `NameError: name '___' is not defined`.
Later cells use your code, so finish each exercise before moving on.

## Part A — Real embeddings, real cosine similarity

`gensim` ships a simple downloader for small pretrained word-vector sets. We'll use
`glove-wiki-gigaword-50` — 50-dimensional vectors trained on Wikipedia text, about 66MB,
small enough to download and load in well under a minute.

GloVe learns its vectors from co-occurrence counts: how often two words appear near each
other, over the whole corpus. Skip-gram, from the lesson, learns them by predicting
neighbors. Both start from the same idea: words that keep similar company get similar
vectors.

In [ ]:
!pip install -q gensim

import gensim.downloader as api

print("Downloading glove-wiki-gigaword-50 (about 66MB)...")
word_vectors = api.load("glove-wiki-gigaword-50")
print("Loaded. Vocabulary size:", len(word_vectors))

## Compare to the toy numbers

The lesson's toy vectors, aunt = (4, 2) and uncle = (3, 3), gave cos(aunt, uncle) ≈ 0.95.
Those values were chosen to keep the arithmetic simple. Real embeddings are learned. The
cell below compares *aunt* and *uncle*, and each of them with *car*, a word used in
different contexts.

### ✏️ Exercise 1 — Cosine similarity by hand

Write `cosine(v1, v2)`, then use it on the real word vectors. The cell prints your
value next to gensim's, and they should agree.

cos(v1, v2) = (v1 · v2) / (|v1| |v2|)

<details><summary><b>Hints</b> (click to open)</summary>

- `np.dot(a, b)` is the dot product; `np.linalg.norm(a)` is the length of `a`.
- `word_vectors["aunt"]` is the 50-dimensional vector for *aunt*.

</details>

In [ ]:
import numpy as np


def cosine(v1, v2):
    return np.dot(___, ___) / (np.linalg.norm(___) * np.linalg.norm(___))


for w1, w2 in [("aunt", "uncle"), ("aunt", "car"), ("uncle", "car")]:
    sim = cosine(word_vectors[w1], word_vectors[w2])
    print(f"cos({w1}, {w2}) = {sim:.3f}   (gensim: {word_vectors.similarity(w1, w2):.3f})")

In [ ]:
# Check for Exercise 1. Run this cell as is.
assert abs(cosine(np.array([4, 2]), np.array([3, 3])) - 0.949) < 1e-3, "lesson toy example: cos((4, 2), (3, 3)) = 0.95"
assert abs(cosine(np.array([4, 2]), np.array([1, 4])) - 0.651) < 1e-3, "cos((4, 2), (1, 4)) = 0.65"
assert abs(cosine(word_vectors["aunt"], word_vectors["uncle"]) - word_vectors.similarity("aunt", "uncle")) < 1e-5
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
import numpy as np


def cosine(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))


for w1, w2 in [("aunt", "uncle"), ("aunt", "car"), ("uncle", "car")]:
    sim = cosine(word_vectors[w1], word_vectors[w2])
    print(f"cos({w1}, {w2}) = {sim:.3f}   (gensim: {word_vectors.similarity(w1, w2):.3f})")
```

</details>

## Nearest neighbors

What is this model's own notion of "close to aunt"?

### ✏️ Exercise 2 — Nearest neighbors by hand

`word_vectors.most_similar` is one line of code. Here you compute the same list
yourself: the cosine similarity of *aunt* to **every** word in the vocabulary at once, as one
matrix operation, then the 8 most similar words.

<details><summary><b>Hints</b> (click to open)</summary>

- `all_vectors @ target` gives one dot product per row.
- `np.linalg.norm(all_vectors, axis=1)` gives one length per row.
- `np.argsort` sorts from smallest to largest. Sorting `-sims` puts the largest first.
- `word_vectors.index_to_key[i]` is the word in row `i`.

</details>

In [ ]:
target = word_vectors["aunt"]
all_vectors = word_vectors.vectors  # one row per vocabulary word
sims = (all_vectors @ ___) / (np.linalg.norm(all_vectors, axis=1) * np.linalg.norm(___))
best_first = np.argsort(___)  # indices, most similar first
my_neighbors = [word_vectors.index_to_key[i] for i in best_first[:9]
                if word_vectors.index_to_key[i] != "aunt"][:8]

for word, score in word_vectors.most_similar("aunt", topn=8):
    print(f"{word:12s} {score:.3f}")
print("\nYours:", my_neighbors)

In [ ]:
# Check for Exercise 2. Run this cell as is.
assert len(sims) == len(word_vectors), "one similarity per vocabulary word"
assert my_neighbors == [w for w, _ in word_vectors.most_similar("aunt", topn=8)], my_neighbors
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
target = word_vectors["aunt"]
all_vectors = word_vectors.vectors
sims = (all_vectors @ target) / (np.linalg.norm(all_vectors, axis=1) * np.linalg.norm(target))
best_first = np.argsort(-sims)
my_neighbors = [word_vectors.index_to_key[i] for i in best_first[:9]
                if word_vectors.index_to_key[i] != "aunt"][:8]

for word, score in word_vectors.most_similar("aunt", topn=8):
    print(f"{word:12s} {score:.3f}")
print("\nYours:", my_neighbors)
```

</details>

## A 2-D map of related words

These embeddings have 50 dimensions (large language models use thousands: GPT-3 uses
12,288). We can't plot
50 dimensions directly, so we reduce to 2 with PCA — a standard technique that finds the 2
directions capturing the most variation, and projects everything onto just those. Look for
groups: family words, vehicle words, and words for places to live.

### ✏️ Exercise 3 — Add a category to the map

Add **four words of a new category** (food, animals, colors, ...) to `WORDS`, then
compute the 2-D PCA coordinates and run the cell. Does the new category form its own
cluster?

<details><summary><b>Hints</b> (click to open)</summary>

- GloVe's vocabulary is lowercase English words, e.g. `"bread"`.
- `PCA(n_components=k)` keeps `k` dimensions.

</details>

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

WORDS = ["aunt", "uncle", "mother", "father", "sister", "brother",
         "car", "bus", "train", "bicycle", "house", "apartment",
         ___, ___, ___, ___]  # four words of a new category, as strings

vectors = np.array([word_vectors[w] for w in WORDS])
coords = PCA(n_components=___).fit_transform(vectors)

plt.figure(figsize=(7, 6))
plt.scatter(coords[:, 0], coords[:, 1])
for word, (x, y) in zip(WORDS, coords):
    plt.annotate(word, (x, y), textcoords="offset points", xytext=(5, 5))
plt.title("50-dimensional word embeddings, projected to 2-D")
plt.show()

In [ ]:
# Check for Exercise 3. Run this cell as is.
assert len(WORDS) >= 16, "add four words"
assert all(w in word_vectors.key_to_index for w in WORDS), [w for w in WORDS if w not in word_vectors.key_to_index]
assert coords.shape == (len(WORDS), 2)
print("✓ Exercise 3 passed")

<details><summary><b>Solution to Exercise 3</b> (open it after you have tried)</summary>

```python
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

WORDS = ["aunt", "uncle", "mother", "father", "sister", "brother",
         "car", "bus", "train", "bicycle", "house", "apartment",
         "bread", "cheese", "apple", "soup"]

vectors = np.array([word_vectors[w] for w in WORDS])
coords = PCA(n_components=2).fit_transform(vectors)

plt.figure(figsize=(7, 6))
plt.scatter(coords[:, 0], coords[:, 1])
for word, (x, y) in zip(WORDS, coords):
    plt.annotate(word, (x, y), textcoords="offset points", xytext=(5, 5))
plt.title("50-dimensional word embeddings, projected to 2-D")
plt.show()
```

</details>

**Reflection:** Look at the scatter plot above. Do family words end up clustered
near each other, and vehicle words clustered separately, even though nobody told the
model what a "family" or "vehicle" category is? If so, where did that structure come
from? GloVe is trained on counts of which words occur near each other. How could those counts
alone produce categories nobody defined? Compare with skip-gram from the lesson.

## Part B — Does Galician actually look distinct to a multilingual model?

This notebook's Galician thread asks: does a general-purpose model actually represent
Galician on its own terms, or does it default to what it has much more data for —
Spanish and Portuguese, both very close relatives? This section probes that directly,
using a small multilingual sentence-embedding model.

In [ ]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

print("Downloading paraphrase-multilingual-MiniLM-L12-v2 (about 470MB)...")
multi_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print("Loaded.")

## The same words, four languages

"Tía miña" is the phrase the attention alignment example works through. Here are its two
words, plus the whole phrase, in Galician, Spanish, Portuguese, and English.

### ✏️ Exercise 4 — Split the embeddings back by language

`multi_model.encode` returns one matrix with all 12 embeddings, in the order of
`flat_texts`. Fill in the loop that hands each language its own 3 rows.

<details><summary><b>Hints</b> (click to open)</summary>

- `i` is the row where the current language starts.
- Each language has `len(texts)` rows.

</details>

In [ ]:
!pip install -q codecarbon
from codecarbon import EmissionsTracker
import time
import numpy as np
import pandas as pd

WORDS_BY_LANGUAGE = {
    "gl": ["tía", "miña", "unha tía miña"],
    "es": ["tía", "mía", "una tía mía"],
    "pt": ["tia", "minha", "uma tia minha"],
    "en": ["aunt", "my", "an aunt of mine"],
}

flat_texts = [text for texts in WORDS_BY_LANGUAGE.values() for text in texts]

tracker_embed = EmissionsTracker(
    project_name="green-embeddings-multilingual",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_embed.start()
start_time = time.time()
embeddings = multi_model.encode(flat_texts, normalize_embeddings=True)
elapsed_embed = time.time() - start_time
emissions_embed_kg = tracker_embed.stop()

vectors_by_language = {}
i = 0
for lang, texts in WORDS_BY_LANGUAGE.items():
    vectors_by_language[lang] = embeddings[i : ___]
    i += ___

print(f"Embedded {len(flat_texts)} texts across 4 languages in {elapsed_embed:.2f}s "
      f"({emissions_embed_kg * 1000:.6f} g CO2eq).")
print("Tiny on its own. Notebook 1 (01a_baseline_measurement) measures costs like this")
print("for four translation approaches.")

In [ ]:
# Check for Exercise 4. Run this cell as is.
assert list(vectors_by_language) == ["gl", "es", "pt", "en"]
assert all(v.shape == (3, embeddings.shape[1]) for v in vectors_by_language.values())
assert np.allclose(vectors_by_language["es"][0], embeddings[3]), "'tía' in Spanish is row 3"
assert np.allclose(vectors_by_language["en"][2], embeddings[-1])
print("✓ Exercise 4 passed")

<details><summary><b>Solution to Exercise 4</b> (open it after you have tried)</summary>

```python
!pip install -q codecarbon
from codecarbon import EmissionsTracker
import time
import numpy as np
import pandas as pd

WORDS_BY_LANGUAGE = {
    "gl": ["tía", "miña", "unha tía miña"],
    "es": ["tía", "mía", "una tía mía"],
    "pt": ["tia", "minha", "uma tia minha"],
    "en": ["aunt", "my", "an aunt of mine"],
}

flat_texts = [text for texts in WORDS_BY_LANGUAGE.values() for text in texts]

tracker_embed = EmissionsTracker(
    project_name="green-embeddings-multilingual",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_embed.start()
start_time = time.time()
embeddings = multi_model.encode(flat_texts, normalize_embeddings=True)
elapsed_embed = time.time() - start_time
emissions_embed_kg = tracker_embed.stop()

vectors_by_language = {}
i = 0
for lang, texts in WORDS_BY_LANGUAGE.items():
    vectors_by_language[lang] = embeddings[i : i + len(texts)]
    i += len(texts)

print(f"Embedded {len(flat_texts)} texts across 4 languages in {elapsed_embed:.2f}s "
      f"({emissions_embed_kg * 1000:.6f} g CO2eq).")
print("Tiny on its own. Notebook 1 (01a_baseline_measurement) measures costs like this")
print("for four translation approaches.")
```

</details>

## Cosine similarity: Galician against the other three

For each of the three items (the two words and the full phrase), how similar is the
Galician embedding to its Spanish, Portuguese, and English counterpart? (Since the vectors
are normalized, a plain dot product *is* the cosine similarity.)

### ✏️ Exercise 5 — Fill the similarity table

For each item, compare the Galician vector with its Spanish, Portuguese, and English
counterparts.

<details><summary><b>Hints</b> (click to open)</summary>

- The vectors were encoded with `normalize_embeddings=True`, so each has length 1.
- For vectors of length 1, the dot product is the cosine similarity.

</details>

In [ ]:
labels = ["tía / aunt", "miña / my", "full phrase"]
rows = []
for idx, label in enumerate(labels):
    gl_vec = vectors_by_language["gl"][idx]
    row = {"item": label}
    for lang in ["es", "pt", "en"]:
        other_vec = vectors_by_language[lang][idx]
        row[f"gl vs {lang}"] = float(np.dot(___, ___))
    rows.append(row)

similarity_df = pd.DataFrame(rows)
similarity_df

In [ ]:
# Check for Exercise 5. Run this cell as is.
assert similarity_df.shape == (3, 4)
_g, _p = vectors_by_language["gl"][0], vectors_by_language["pt"][0]
_expected = float(np.dot(_g, _p) / (np.linalg.norm(_g) * np.linalg.norm(_p)))
assert abs(similarity_df.loc[0, "gl vs pt"] - _expected) < 1e-5
print("✓ Exercise 5 passed")

<details><summary><b>Solution to Exercise 5</b> (open it after you have tried)</summary>

```python
labels = ["tía / aunt", "miña / my", "full phrase"]
rows = []
for idx, label in enumerate(labels):
    gl_vec = vectors_by_language["gl"][idx]
    row = {"item": label}
    for lang in ["es", "pt", "en"]:
        other_vec = vectors_by_language[lang][idx]
        row[f"gl vs {lang}"] = float(np.dot(gl_vec, other_vec))
    rows.append(row)

similarity_df = pd.DataFrame(rows)
similarity_df
```

</details>

## Visualizing the gap

The table above has the numbers; a bar chart makes the pattern easier to see at a
glance, and a 2-D map of the actual embeddings shows *why* the numbers come out that
way — literally where each language's version of these phrases sits in vector space.

In [ ]:
import matplotlib.pyplot as plt

similarity_df.set_index("item").plot(kind="bar", figsize=(7, 5), ylim=(0, 1))
plt.ylabel("cosine similarity to Galician")
plt.title("How close does each language sit to Galician?")
plt.xticks(rotation=0)
plt.legend(title="compared to")
plt.tight_layout()
plt.show()

## A 2-D map of the same 12 embeddings

Same PCA idea as Part A: project the real 384-dimensional embeddings down to 2
dimensions, and plot all 12 (word/phrase, language) pairs together, colored by
language.

In [ ]:
from sklearn.decomposition import PCA

point_langs = [lang for lang, texts in WORDS_BY_LANGUAGE.items() for _ in texts]
point_labels = [text for texts in WORDS_BY_LANGUAGE.values() for text in texts]
lang_names = {"gl": "Galician", "es": "Spanish", "pt": "Portuguese", "en": "English"}
colors = {"gl": "tab:blue", "es": "tab:orange", "pt": "tab:green", "en": "tab:red"}

coords = PCA(n_components=2).fit_transform(embeddings)

plt.figure(figsize=(7, 6))
for lang in WORDS_BY_LANGUAGE:
    mask = [l == lang for l in point_langs]
    plt.scatter(coords[mask, 0], coords[mask, 1], label=lang_names[lang], color=colors[lang])
for (x, y), text in zip(coords, point_labels):
    plt.annotate(text, (x, y), textcoords="offset points", xytext=(5, 5), fontsize=8)
plt.title("Same phrases, four languages, projected to 2-D")
plt.legend()
plt.tight_layout()
plt.show()

**Reflection:** Compare the "gl vs es", "gl vs pt" and "gl vs en" numbers in the table
above. Are any of them low? This model was trained to put translations close together: what
values would you expect? Which item gets exactly 1.00, and why? In the 2-D map, are the
points grouped by language or by meaning? What does a similarity of 1.00 tell you about
how much this model can know about Galician as its own language?

## Code literacy check

Before you move on, pick at least two code cells above that you actually ran and read
today, and add `#` comments to them explaining, in your own words, what each meaningful
line or block does and *why* it's there, in language a classmate who hasn't seen the code
could follow.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini — whichever you
   have access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, go check the code again —
   who's right?

This is a self-check on your own understanding: a way to catch, in your own words, where
your mental model of the code breaks down against a second reader.